In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 4, 5, 10]

In [ ]:
def prepare_gaslight_the_monitor_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    gaslight_the_monitor_df = load_monitoring_data(base_dir=base_dir)
    for i in range(1, 4):
        gaslight_the_monitor_df[f'response_{i}_gaslight'] = gaslight_the_monitor_df.data.apply(
            lambda x: extract_verdict(x[f'response_{i}'], tag_word='verdict')
        )
    gaslight_the_monitor_df['average_suspiciousness_gaslight'] = (gaslight_the_monitor_df['response_1_gaslight'] + gaslight_the_monitor_df['response_2_gaslight'] + gaslight_the_monitor_df['response_3_gaslight']) / 3
    gaslight_the_monitor_df['max_suspiciousness_gaslight'] = np.max([gaslight_the_monitor_df['response_1_gaslight'], gaslight_the_monitor_df['response_2_gaslight'], gaslight_the_monitor_df['response_3_gaslight']], axis=0)
    gaslight_the_monitor_df['min_suspiciousness_gaslight'] = np.min([gaslight_the_monitor_df['response_1_gaslight'], gaslight_the_monitor_df['response_2_gaslight'], gaslight_the_monitor_df['response_3_gaslight']], axis=0)
    return gaslight_the_monitor_df

def merge_and_compute_features_for_gaslight_the_monitor(combined_results: pd.DataFrame, 
                             gaslight_the_monitor_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        gaslight_the_monitor_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features

    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    
    return merged_df


In [ ]:
if __name__ == "__main__":
    include_failed_attempts = False
    include_failed_main_task = False
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    gaslight_the_monitor_df = prepare_gaslight_the_monitor_data(base_dir='monitor_responses/gaslight_the_monitor', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
    
    #%%
    give_hint = True
    merged_df = merge_and_compute_features_for_gaslight_the_monitor(combined_results, gaslight_the_monitor_df)
    merged_df['1_minus_2'] = merged_df['response_1_gaslight'] - merged_df['response_2_gaslight']
    merged_df['1_minus_3'] = merged_df['response_1_gaslight'] - merged_df['response_3_gaslight']
    merged_df['2_minus_3'] = merged_df['response_2_gaslight'] - merged_df['response_3_gaslight']
    merged_df['max_1_and_2'] = np.max([merged_df['response_1_gaslight'], merged_df['response_2_gaslight']], axis=0)
    merged_df['max_1_and_3'] = np.max([merged_df['response_1_gaslight'], merged_df['response_3_gaslight']], axis=0)
    merged_df['max_2_and_3'] = np.max([merged_df['response_2_gaslight'], merged_df['response_3_gaslight']], axis=0)
    plot_roc_curves(merged_df, combined_results, give_hint=give_hint, include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, monitors = ['gpt-4o'], metrics = ['response_1_gaslight', 'response_2_gaslight', 'response_3_gaslight', 'average_suspiciousness_gaslight', 'max_suspiciousness_gaslight', 'min_suspiciousness_gaslight'])
    # plot_suspiciousness_distributions(merged_df, monitors = ['gpt-4o'], metrics = ['response_1_gaslight', 'response_2_gaslight', 'response_3_gaslight', 'average_suspiciousness_gaslight', 'max_suspiciousness_gaslight'])
    # %%
    
    merged_df[(merged_df['response_1_gaslight'] < merged_df['response_2_gaslight']) & (merged_df['response_2_gaslight'] < merged_df['response_3_gaslight'])].transcript_type.value_counts()
    #%%
    merged_df.transcript_type.value_counts()
